# 🚀 Compilation LLM Simple pour Godot

**GDExtension minimale pour utiliser llama.cpp dans Godot**

## Caractéristiques
- ✅ Code ultra simple (~300 lignes)
- ✅ Utilise llama.cpp officiel
- ✅ Compilation rapide (5-10 min)
- ✅ Interface claire: `load_model()`, `generate()`

## Structure du ZIP à uploader
```
llm_simple_sources.tar.gz contenant:
├── llm_simple/
│   ├── include/llm_simple.h
│   ├── src/llm_simple.cpp
│   ├── src/register_types.cpp
│   ├── src/register_types.h
│   └── CMakeLists.txt
├── addons/llm_simple/llm_simple.gdextension
└── godot-cpp/  (submodule cloné)
```

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 1: Installation Outils
# ═══════════════════════════════════════════════════════════════

print("🔧 Installation des outils...\n")

!apt-get update -qq
!apt-get install -y -qq mingw-w64 cmake ninja-build zip unzip file wget
!pip install -q scons

print("\n✅ Outils installés!")
!x86_64-w64-mingw32-g++ --version | head -1

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 2: Upload Sources
# ═══════════════════════════════════════════════════════════════

from google.colab import files
import os
import subprocess

print("📤 Uploadez llm_simple_sources.tar.gz\n")

uploaded = files.upload()

if not uploaded:
    raise Exception("Aucun fichier uploadé!")

filename = list(uploaded.keys())[0]
print(f"\n📦 Fichier: {filename}")
print(f"   Taille: {len(uploaded[filename]) / 1024 / 1024:.2f} MB\n")

# Extraction
!rm -rf /content/workspace
!mkdir -p /content/workspace
!tar -xzf {filename} -C /content/workspace

print("\n📂 Contenu:")
!ls -la /content/workspace

# Vérification
checks = [
    "/content/workspace/llm_simple/src/llm_simple.cpp",
    "/content/workspace/llm_simple/CMakeLists.txt",
    "/content/workspace/godot-cpp"
]

for path in checks:
    exists = os.path.exists(path)
    print(f"{'✅' if exists else '❌'} {path}")
    if not exists:
        raise Exception(f"Fichier manquant: {path}")

print("\n✅ Structure OK!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 3: Télécharger llama.cpp Précompilé
# ═══════════════════════════════════════════════════════════════

import os
os.chdir("/content/workspace")

print("📥 Téléchargement llama.cpp release officielle...\n")

# Télécharger release Windows
LLAMA_VERSION = "b4357"
LLAMA_URL = f"https://github.com/ggerganov/llama.cpp/releases/download/{LLAMA_VERSION}/llama-{LLAMA_VERSION}-bin-win-avx2-x64.zip"

!wget -q --show-progress {LLAMA_URL} -O llama_bin.zip
!unzip -q llama_bin.zip -d llama_bin

print("\n📦 Contenu llama.cpp:")
!ls -lh llama_bin/

# Vérifier les DLLs
if os.path.exists("llama_bin/llama.dll") and os.path.exists("llama_bin/ggml.dll"):
    print("\n✅ DLLs llama.cpp trouvées!")
else:
    print("\n⚠️  Structure différente, recherche des DLLs...")
    !find llama_bin -name "*.dll"

# Cloner repo pour les headers
print("\n📥 Clonage llama.cpp pour headers...")
!git clone --depth 1 --branch {LLAMA_VERSION} https://github.com/ggerganov/llama.cpp.git

print("\n✅ llama.cpp prêt!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 4: Compilation godot-cpp
# ═══════════════════════════════════════════════════════════════

import os
os.chdir("/content/workspace/godot-cpp")

print("🔨 Compilation godot-cpp (5-10 min)...\n")

!scons platform=windows target=template_release arch=x86_64 use_mingw=yes -j$(nproc) 2>&1 | tail -30

# Vérification
libs = !find bin -name "*.a" 2>/dev/null
if libs and libs[0]:
    print("\n✅ godot-cpp compilé!")
    !ls -lh bin/*.a
else:
    raise Exception("godot-cpp compilation échouée")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 5: Compilation llm_simple
# ═══════════════════════════════════════════════════════════════

import os
os.chdir("/content/workspace/llm_simple")

print("🔨 Compilation llm_simple (2-3 min)...\n")

# Créer un toolchain file MinGW
with open("/tmp/mingw-toolchain.cmake", "w") as f:
    f.write("""set(CMAKE_SYSTEM_NAME Windows)
set(CMAKE_C_COMPILER x86_64-w64-mingw32-gcc)
set(CMAKE_CXX_COMPILER x86_64-w64-mingw32-g++)
set(CMAKE_RC_COMPILER x86_64-w64-mingw32-windres)
set(CMAKE_FIND_ROOT_PATH /usr/x86_64-w64-mingw32)
set(CMAKE_FIND_ROOT_PATH_MODE_PROGRAM NEVER)
set(CMAKE_FIND_ROOT_PATH_MODE_LIBRARY ONLY)
set(CMAKE_FIND_ROOT_PATH_MODE_INCLUDE ONLY)
""")

# Créer répertoire de build
!rm -rf build && mkdir build && cd build

# Compiler avec liaison statique des DLLs llama.cpp
!cd build && x86_64-w64-mingw32-g++ -shared \
    -o llm_simple.dll \
    ../src/llm_simple.cpp \
    ../src/register_types.cpp \
    -I../include \
    -I../../godot-cpp/include \
    -I../../godot-cpp/gen/include \
    -I../../llama.cpp/include \
    -I../../llama.cpp/ggml/include \
    -L../../godot-cpp/bin \
    -L../../llama_bin \
    -lgodot-cpp.windows.template_release.x86_64 \
    -lllama \
    -lggml \
    -static-libgcc \
    -static-libstdc++ \
    -std=c++17

# Vérification
if os.path.exists("build/llm_simple.dll"):
    size = os.path.getsize("build/llm_simple.dll") / 1024 / 1024
    print(f"\n✅ llm_simple.dll compilée! ({size:.2f} MB)")
else:
    raise Exception("DLL non générée!")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELLULE 6: Packaging et Téléchargement
# ═══════════════════════════════════════════════════════════════

from google.colab import files
import os

print("📦 Création du package final...\n")

os.chdir("/content")
!mkdir -p output/addons/llm_simple/bin

# Copier DLL extension
!cp workspace/llm_simple/build/llm_simple.dll output/addons/llm_simple/bin/llm_simple.windows.template_release.x86_64.dll

# Copier DLLs llama.cpp
!cp workspace/llama_bin/*.dll output/addons/llm_simple/bin/ 2>/dev/null || echo "Note: DLLs dans sous-dossier"
!find workspace/llama_bin -name "*.dll" -exec cp {} output/addons/llm_simple/bin/ \;

# Copier .gdextension
!cp workspace/addons/llm_simple/llm_simple.gdextension output/addons/llm_simple/

# README
with open("output/README.txt", "w") as f:
    f.write("""═══════════════════════════════════════════════════════════
  LLM SIMPLE pour Godot - Installation
═══════════════════════════════════════════════════════════

INSTALLATION
============

1. Copier le dossier 'addons/llm_simple' dans votre projet Godot:
   c:\\Users\\PGNK2128\\Godot-MCP\\addons\\llm_simple\\

2. Télécharger un modèle GGUF (ex: Phi-3-mini-3B):
   https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf

3. Placer le modèle dans:
   c:\\Users\\PGNK2128\\Godot-MCP\\models\\phi-3-mini-3b.gguf

4. Redémarrer Godot

UTILISATION
===========

extends Node

@onready var llm = LLMSimple.new()

func _ready():
    add_child(llm)
    
    if llm.load_model("res://models/phi-3-mini-3b.gguf", 8192):
        var response = llm.generate("Bonjour! Qui es-tu?", 256)
        print(response)

API
===

- load_model(path: String, context_size: int = 8192) -> bool
- generate(prompt: String, max_tokens: int = 256) -> String
- unload_model()
- is_model_loaded() -> bool
- get_last_error() -> String

SIGNAUX
=======

- generation_started()
- generation_finished(text: String)
- generation_error(error: String)

═══════════════════════════════════════════════════════════
""")

# Liste des fichiers
print("📂 Contenu du package:\n")
!find output -type f

# Créer ZIP
!zip -r llm_simple_addon.zip output/

print("\n" + "="*60)
print("✅ COMPILATION RÉUSSIE!")
print("="*60)
print("\n📥 Téléchargement...\n")

files.download("llm_simple_addon.zip")

print("\n🎉 Terminé!")
print("\nInstallez l'addon et téléchargez un modèle GGUF.")